In [ ]:
pip install mp-api

In [2]:
from mp_api.client import MPRester

# You can pass your key directly, or set it as the MP_API_KEY environment variable
api_key = "auNZYZfJjGJxC7GXjYXgUDRqh8Ek6dYk"

with MPRester(api_key) as mpr:
    # Query the summary endpoint for Silicon (mp-149)
    results = mpr.summary.search(material_ids=["mp-149"])
    
    if results:
        silicon = results[0]
        print(f"Formula: {silicon.formula_pretty}")
        print(f"Band Gap: {silicon.band_gap} eV")
        print(f"Volume: {silicon.volume}")

c:\Users\sk191\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sk191\AppData\Local\Temp\ipykernel_21020\1570379034.py:8: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  results = mpr.summary.search(material_ids=["mp-149"])
Retrieving SummaryDoc documents: 100%|██████████| 1/1 [00:00<?, ?it/s]

Formula: Si
Band Gap: 0.6105 eV
Volume: 40.32952684741405


In [ ]:
pip install openpyxl

In [6]:
from mp_api.client import MPRester
import pandas as pd

# 1. Insert your Materials Project API key here
API_KEY = "auNZYZfJjGJxC7GXjYXgUDRqh8Ek6dYk"

def download_oxide_perovskites_to_excel():
    print("Connecting to Materials Project...")
    
    with MPRester(API_KEY) as mpr:
        # 2. Search for Oxide Perovskite candidates without the 'limit' kwarg
        # formula="ABC3" ensures a 1:1:3 atomic ratio.
        # elements=["O"] ensures Oxygen is one of those elements.
        print("Querying database (this might take a moment to fetch the matches)...")
        results = mpr.summary.search(
            formula="ABC3", 
            elements=["O"]
        )
        
    # 3. Keep only the first 1000 results
    results = results[:1000]
    print(f"Successfully filtered down to {len(results)} oxide perovskite candidates.")
    
    # 4. Parse the results into a list of dictionaries
    data = []
    for doc in results:
        comp = doc.composition.reduced_composition
        # Double check to ensure Oxygen is exactly 60% of the atoms (3 out of 5)
        if comp.get_atomic_fraction("O") != 0.6: 
            continue
            
        material_dict = {
            "Material_ID": str(doc.material_id),
            "Formula": doc.formula_pretty,
            "Volume": doc.volume,
            "Density": doc.density,
            "Band_Gap_eV": doc.band_gap,
            "Is_Stable": doc.is_stable,
            "Formation_Energy_per_atom": doc.formation_energy_per_atom,
            "Energy_Above_Hull": doc.energy_above_hull,
            "Crystal_System": str(doc.symmetry.crystal_system) if doc.symmetry else None,
            "Space_Group": doc.symmetry.symbol if doc.symmetry else None,
            "Theoretical": doc.theoretical
        }
        data.append(material_dict)
        
    # 5. Convert to a Pandas DataFrame
    df = pd.DataFrame(data)
    
    # 6. Export to Excel
    output_filename = "Metal_Oxide_Perovskites_1000.xlsx"
    df.to_excel(output_filename, index=False)
    print(f"Data successfully saved to {output_filename}! Total saved: {len(df)}")

if __name__ == "__main__":
    download_oxide_perovskites_to_excel()

Connecting to Materials Project...
Querying database (this might take a moment to fetch the matches)...


C:\Users\sk191\AppData\Local\Temp\ipykernel_21020\4257319387.py:15: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  results = mpr.summary.search(
Retrieving SummaryDoc documents: 100%|██████████| 2672/2672 [00:14<00:00, 182.15it/s]


Successfully filtered down to 1000 oxide perovskite candidates.
Data successfully saved to Metal_Oxide_Perovskites_1000.xlsx! Total saved: 875


In [ ]:
pip install torch

In [2]:
import pandas as pd
import torch 
import torch.nn as nn
from pymatgen.core import Composition

In [4]:
import pandas as pd
import torch
import torch.nn as nn
from pymatgen.core import Composition

# ==========================================
# 1. Setup the Embedding Layer
# ==========================================
embed_dim = 8 # Change this if you want larger vectors (e.g., 32, 64)
embedding_layer = nn.Embedding(num_embeddings=119, embedding_dim=embed_dim)

# If you have pre-trained weights, uncomment the line below and load them:
# embedding_layer.load_state_dict(torch.load("trained_weights.pth"))

def embed_formula(formula):
    """Parses a formula and returns its weighted embedding vector as a list."""
    try:
        # Check for empty cells
        if pd.isna(formula) or str(formula).strip() == "":
            return None
            
        comp = Composition(str(formula).strip())
        atom_ids = []
        atom_fractions = []
        
        for element, fraction in comp.fractional_composition.items():
            atom_ids.append(element.Z) # Atomic number
            atom_fractions.append(fraction)
            
        # Convert to PyTorch tensors
        ids_tensor = torch.tensor(atom_ids)
        fractions_tensor = torch.tensor(atom_fractions, dtype=torch.float32).unsqueeze(-1)
        
        with torch.no_grad():
            embeds = embedding_layer(ids_tensor)
            weighted_embeds = embeds * fractions_tensor
            formula_vector = torch.sum(weighted_embeds, dim=0)
            
        return [round(x, 4) for x in formula_vector.tolist()]
    
    except Exception as e:
        # Returns None if the formula is unreadable by pymatgen
        return None

# ==========================================
# 2. Load the CSV File
# ==========================================
file_name = "Metal_Oxide_Perovskites_1000.csv"
print(f"Loading '{file_name}'...")
df = pd.read_csv(file_name)

# Automatically select the FIRST column (index 0)
formula_col_name = df.columns[0]
print(f"Applying embeddings to the first column: '{formula_col_name}'")

# ==========================================
# 3. Apply the Embedding
# ==========================================
df['formula_embedding'] = df[formula_col_name].apply(embed_formula)

# ==========================================
# 4. Save the Result
# ==========================================
output_filename = "Metal_Oxide_Perovskites_1000_embedded_output.csv"
df.to_csv(output_filename, index=False)

print(f"Success! The embedded vectors have been saved to '{output_filename}'.")

Loading 'Metal_Oxide_Perovskites_1000.csv'...
Applying embeddings to the first column: 'Formula'
Success! The embedded vectors have been saved to 'Metal_Oxide_Perovskites_1000_embedded_output.csv'.


In [6]:
import pandas as pd

# 1. Load your dataset
df = pd.read_csv('Metal_Oxide_Perovskites_1000_embedded_output.csv')

# 2. Drop the text/string columns that we cannot do direct math on
columns_to_drop = ['Formula', 'formula_embedding']
df_numeric = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# 3. Force all remaining columns to be numbers
# (errors='coerce' turns typos like the dash in row 4 into a blank 'NaN' so math doesn't crash)
df_numeric = df_numeric.apply(pd.to_numeric, errors='coerce')

# 4. Now that the data is clean, separate features and target
# Features: Everything except the last column
features = df_numeric.iloc[:, :-1]

# Target: The very last column (Band_Gap_eV)
target = df_numeric.iloc[:, -1]

# 5. Calculate the correlation
correlations = features.corrwith(target)

# 6. Display sorted results
print(f"Correlation with '{target.name}':\n")
print(correlations.sort_values(ascending=False))

Correlation with 'Band_Gap_eV':

Volume                       0.158018
Energy_Above_Hull           -0.244213
Density                     -0.270763
Formation_Energy_per_atom   -0.285098
dtype: float64


In [7]:
import pandas as pd
df = pd.read_csv('Metal_Oxide_Perovskites_1000_embedded_output.csv')
print(df.columns)   

Index(['Formula', 'Volume', 'Density', 'Formation_Energy_per_atom',
       'Energy_Above_Hull', 'Band_Gap_eV', 'formula_embedding'],
      dtype='str')


In [8]:
df.head(5)

,Formula,Volume,Density,Formation_Energy_per_atom,Energy_Above_Hull,Band_Gap_eV,formula_embedding
0,TiCdO3,277.936103,2.488707,-1.556926,1.114535,0.0000,"[-0.6652, -0.0712, -0.04, -1.2376, 0.3517, 1.0..."
1,NaNO3,268.855749,2.099817,-1.350257,0.012940,2.7742,"[-0.1235, -0.2686, -0.0138, -1.1599, -0.111, 0..."
2,VHO3,262.154142,2.532357,-1.922323,0.063993,0.0094,"[0.2203, -0.3718, -0.4327, -1.0186, -0.0977, 1..."
3,SrTeO3,2270.903793,4.619314,-2.377819,0.003206,3.2315,"[-0.304, -0.7513, -0.5039, -1.0024, -0.168, 1...."
4,NaBrO3,437.189796,2.292478,-0.299051,0.700011,0.0000,"[-0.2229, 0.2422, 0.171, -1.2023, 0.2959, 1.32..."


In [9]:
pip install scikit-learn pandas numpy

  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# ==========================================
# 1. Load and Clean the Data
# ==========================================
print("Loading data...")
# Replace with your actual filename
df = pd.read_csv('Metal_Oxide_Perovskites_1000_embedded_output.csv')

# Convert the 'Volume' column to pure numbers (turns weird symbols like '-' into NaN)
df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')
df['Band_Gap_eV'] = pd.to_numeric(df['Band_Gap_eV'], errors='coerce')

# Drop rows where Volume or Band Gap is missing
df = df.dropna(subset=['Volume', 'Band_Gap_eV'])

# ==========================================
# 2. Unpack the Embeddings
# ==========================================
print("Unpacking embeddings...")
# Convert the string representation of the list back into an actual Python list
df['formula_embedding'] = df['formula_embedding'].apply(ast.literal_eval)

# Split the list into separate columns (embed_0, embed_1, embed_2, etc.)
embed_df = pd.DataFrame(df['formula_embedding'].tolist(), index=df.index)
embed_df.columns = [f'embed_{i}' for i in range(embed_df.shape[1])]

# Attach these new embedding columns back to the main dataframe
df = pd.concat([df, embed_df], axis=1)

# ==========================================
# 3. Define Features (X) and Target (y)
# ==========================================
# We use Volume PLUS all the new embedding columns
feature_cols = feature_cols = ['Density', 'Formation_Energy_per_atom', 'Energy_Above_Hull'] + list(embed_df.columns)
X = df[feature_cols]
y = df['Band_Gap_eV']

# Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 4. Train and Evaluate Multiple Models
# ==========================================
print("\nTraining models and comparing results...\n")
print("-" * 50)
print(f"{'Algorithm':<25} | {'R² Score':<10} | {'RMSE':<10}")
print("-" * 50)

# Dictionary of models to test
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Support Vector Machine": SVR(kernel='rbf')
}

best_model_name = ""
best_r2 = -float('inf')

for name, model in models.items():
    # Train the AI
    model.fit(X_train, y_train)
    
    # Make predictions on the unseen test data
    predictions = model.predict(X_test)
    
    # Calculate metrics
    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    
    # Print results
    print(f"{name:<25} | {r2:>8.4f}   | {rmse:>8.4f}")
    
    # Track the best model based on R2
    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name

print("-" * 50)
print(f"\nBest Performing Model: **{best_model_name}**")

Loading data...
Unpacking embeddings...

Training models and comparing results...

--------------------------------------------------
Algorithm                 | R² Score   | RMSE      
--------------------------------------------------
Linear Regression         |   0.2800   |   1.3467
Random Forest             |   0.6102   |   0.9908
Gradient Boosting         |   0.5426   |   1.0733
Support Vector Machine    |  -0.0686   |   1.6406
--------------------------------------------------

Best Performing Model: **Random Forest**
